# 02_silver — Clean the partner data with PySpark

Silver turns the raw text in Bronze into clean, typed, de-duplicated data with data quality flags.

| Problem in Bronze | Example | Silver fix |
|---|---|---|
| Extra spaces | ` BNK202610120004 ` | Trim and collapse spaces |
| Mixed case | `cleared`, `Cleared `, `returned` | Upper case for codes, standard names for providers and shops |
| Currency symbols and commas | `$1,250.50`, `4,500.00` | Remove `$` and `,`, convert to decimal |
| Missing or placeholder values | blank, `N/A` | Convert to null and flag `MISSING_...` or `INVALID_...` |
| Different date formats | `2026-10-11`, `10/11/2026`, `11-Oct-2026` | Try each format |
| Business rule violations | negative amount, `PENDING`, future date, approved > billed, total ≠ labor + parts | Flag in `dq_errors` |
| Duplicates and updates | same row twice, status changed on a later day | Keep the latest version, count versions |

**Rules used in this notebook**
- Invalid rows are kept and flagged (`dq_status = INVALID`), never silently dropped.
- `dq_errors` make a row invalid; `dq_warnings` are kept for review but the row stays valid.
- Silver is rebuilt from Bronze on every run.

## Step 1: Helper functions

**Why:** the three partners need the same cleaning rules. Writing each rule once as a function keeps the rules identical for every partner and keeps the partner cells short.

`try_cast` and `try_to_timestamp` return null for a value that cannot be converted. Without them, one bad value such as `N/A` would stop the whole job.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def clean_text(col_name):
    """Trim, replace repeated spaces with one space, and turn empty text into null."""
    c = F.regexp_replace(F.trim(F.col(col_name)), r"\s+", " ")
    return F.when(c == "", None).otherwise(c)

def clean_code(col_name):
    """Clean text and convert to upper case. Used for IDs and status codes."""
    return F.upper(clean_text(col_name))

def to_int(col_name):
    """Convert text to an integer. Blank, N/A, or any non-number becomes null."""
    return F.expr(f"try_cast(trim({col_name}) AS INT)")

def to_amount(col_name):
    """Remove $, commas, and spaces, then convert to DECIMAL(12,2). Invalid values become null."""
    return F.expr(f"try_cast(regexp_replace({col_name}, '[$, ]', '') AS DECIMAL(12,2))")

def to_date_multi(col_name):
    """Parse the date formats partners send: 2026-10-11, 10/11/2026, 11-Oct-2026."""
    c = f"trim({col_name})"
    return F.coalesce(
        F.expr(f"to_date(try_to_timestamp({c}, 'yyyy-MM-dd'))"),
        F.expr(f"to_date(try_to_timestamp({c}, 'MM/dd/yyyy'))"),
        F.expr(f"to_date(try_to_timestamp({c}, 'dd-MMM-yyyy'))"),
    )

def file_date():
    """Read the date from the file name, for example BANK_20261012.csv -> 2026-10-12."""
    return F.expr("to_date(try_to_timestamp(regexp_extract(_source_file, '([0-9]{8})', 1), 'yyyyMMdd'))")

def issue_list(*checks):
    """Build a list of issue codes from (condition, code) pairs. Only failed checks are kept."""
    arr = F.array(*[F.when(cond, F.lit(code)) for cond, code in checks])
    return F.filter(arr, lambda x: x.isNotNull())

def keep_latest(df, key_col):
    """Keep one row per key: the newest file first, then the row with the fewest errors."""
    w = Window.partitionBy(key_col).orderBy(
        F.col("file_date").desc(),
        F.size("dq_errors").asc(),
        F.col("_ingested_at").desc(),
    )
    return (df
            .withColumn("record_versions", F.count(F.lit(1)).over(Window.partitionBy(key_col)))
            .withColumn("_rn", F.row_number().over(w))
            .filter(F.col("_rn") == 1)
            .drop("_rn"))

def finish(df):
    """Add the data quality status and the load time."""
    return (df
            .withColumn("dq_status", F.when(F.size("dq_errors") == 0, "VALID").otherwise("INVALID"))
            .withColumn("_silver_loaded_at", F.current_timestamp()))

## Step 2: Profile the raw data

**Why:** look at the problems before fixing them. This shows every status value received and every amount and date that is not in the expected format.

In [0]:
bank_bronze = spark.table("insurance.bronze.bank_settlement")

print("Distinct status values received:")
bank_bronze.groupBy("status").count().orderBy("status").show(truncate=False)

print("Amount values that are not plain numbers:")
bank_bronze.filter(~F.col("amount").rlike(r"^-?[0-9]+(\.[0-9]+)?$") | F.col("amount").isNull()) \
    .select("bank_ref", "amount", "_source_file").show(truncate=False)

print("Date values that are not in yyyy-MM-dd format:")
bank_bronze.filter(~F.col("settlement_date").rlike(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$") | F.col("settlement_date").isNull()) \
    .select("bank_ref", "settlement_date", "_source_file").show(truncate=False)

## Step 3: BANK → Silver

### 3.1. Clean and convert every column, keeping a cleaned copy of the raw text for the checks

**Why:** every Bronze column is text. Each column is converted to the right type: IDs and status are trimmed and set to upper case, `$` and commas are removed from amounts, `N/A` and blanks become null, and the three date formats partners send are parsed. A cleaned copy of the raw text (`_..._raw`) is kept only to tell the difference between a value that was **missing** and a value that was **present but invalid**.

In [0]:
VALID_BANK_STATUS = ["CLEARED", "RETURNED"]

bank_bronze = spark.table("insurance.bronze.bank_settlement")

bank = bank_bronze.select(
    clean_code("bank_ref").alias("bank_ref"),
    to_int("payment_id").alias("payment_id"),
    to_int("claim_id").alias("claim_id"),
    to_amount("amount").alias("amount"),
    to_date_multi("settlement_date").alias("settlement_date"),
    clean_code("status").alias("status"),
    clean_text("claim_id").alias("_claim_id_raw"),
    clean_text("amount").alias("_amount_raw"),
    clean_text("settlement_date").alias("_date_raw"),
    "_source_file",
    "_ingested_at",
)

### 3.2. Add business columns

**Why:** reporting needs columns that are not in the file. `source_system` identifies the partner, `file_date` records which night the row arrived (taken from the file name), `settlement_month` is used for monthly totals, and `is_returned` makes returned payments easy to filter.

In [0]:
bank = (bank
    .withColumn("source_system", F.lit("BANK"))
    .withColumn("file_date", file_date())
    .withColumn("settlement_month", F.date_format("settlement_date", "yyyy-MM"))
    .withColumn("is_returned", F.col("status") == "RETURNED"))

### 3.3. Data quality checks

**Why:** rows with problems are **kept and flagged**, not deleted. Each failed rule adds a code to `dq_errors`. Any error makes the row `INVALID`. Gold uses only `VALID` rows, and the operations team uses the error codes to follow up with the partner.

In [0]:
bank = bank.withColumn("dq_errors", issue_list(
    (F.col("bank_ref").isNull(),                                          "MISSING_BANK_REF"),
    (F.col("_claim_id_raw").isNull(),                                     "MISSING_CLAIM_ID"),
    (F.col("_claim_id_raw").isNotNull() & F.col("claim_id").isNull(),     "INVALID_CLAIM_ID"),
    (F.col("_amount_raw").isNull(),                                       "MISSING_AMOUNT"),
    (F.col("_amount_raw").isNotNull() & F.col("amount").isNull(),         "INVALID_AMOUNT"),
    (F.col("amount") < 0,                                                 "NEGATIVE_AMOUNT"),
    (F.col("_date_raw").isNull(),                                         "MISSING_SETTLEMENT_DATE"),
    (F.col("_date_raw").isNotNull() & F.col("settlement_date").isNull(),  "INVALID_SETTLEMENT_DATE"),
    (~F.col("status").isin(VALID_BANK_STATUS) | F.col("status").isNull(), "INVALID_STATUS"),
)).withColumn("dq_warnings", F.array().cast("array<string>"))

### 3.4. Remove duplicates and keep the latest version of each payment

**Why:** Bronze contains every version of a payment. An exact duplicate row appears twice in the Day 2 file, and `BNK202610110003` was CLEARED on Day 1 and RETURNED on Day 2. One row per `bank_ref` is kept: the newest file first, then the row with the fewest errors. `record_versions` shows how many times the partner sent that reference.

In [0]:
bank = keep_latest(bank, "bank_ref")

### 3.5. Final columns

**Why:** only the business columns, data quality columns, and lineage columns are kept. Silver is fully rebuilt from Bronze on every run (`overwrite`), so a change in a cleaning rule is applied to all history.

In [0]:
bank_silver = finish(bank).select(
    "source_system", "bank_ref", "payment_id", "claim_id", "amount", "settlement_date",
    "settlement_month", "status", "is_returned", "file_date", "record_versions",
    "dq_status", "dq_errors", "dq_warnings", "_source_file", "_ingested_at", "_silver_loaded_at",
)

bank_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("insurance.silver.bank_settlement")
display(spark.table("insurance.silver.bank_settlement").orderBy("file_date", "bank_ref"))

## Step 4: TPA → Silver

### 4.1. Clean and convert

**Why:** the same cleaning rules as BANK, applied to TPA columns. `billed_amount` values such as `4,500.00` contain thousands separators, and service dates arrive in different formats.

In [0]:
tpa_bronze = spark.table("insurance.bronze.tpa_settlement")

# Reference list of approved provider names (in production this is a reference table)
provider_ref = spark.createDataFrame([
    ("MEMORIAL HERMANN",     "Memorial Hermann"),
    ("BAYLOR SCOTT & WHITE", "Baylor Scott & White"),
    ("HCA HOUSTON",          "HCA Houston"),
    ("METHODIST HOSPITAL",   "Methodist Hospital"),
    ("CARENOW URGENT CARE",  "CareNow Urgent Care"),
], ["provider_key", "provider_name_std"])

tpa = tpa_bronze.select(
    clean_code("tpa_ref").alias("tpa_ref"),
    to_int("claim_id").alias("claim_id"),
    clean_text("provider_name").alias("provider_name_clean"),
    clean_code("provider_name").alias("provider_key"),
    to_amount("billed_amount").alias("billed_amount"),
    to_amount("approved_amount").alias("approved_amount"),
    to_date_multi("service_date").alias("service_date"),
    clean_text("claim_id").alias("_claim_id_raw"),
    clean_text("billed_amount").alias("_billed_raw"),
    clean_text("approved_amount").alias("_approved_raw"),
    clean_text("service_date").alias("_date_raw"),
    "_source_file",
    "_ingested_at",
)

### 4.2. Standardise provider names using the reference list

**Why:** the same provider arrives as `Memorial Hermann`, `memorial hermann`, and ` MEMORIAL HERMANN `. Names are matched against a reference list of approved provider names. `initcap` alone would turn `HCA Houston` into `Hca Houston`, so a reference list is used. Names not in the list are kept in title case and flagged with the warning `UNKNOWN_PROVIDER`.

In [0]:
tpa = (tpa.join(provider_ref, on="provider_key", how="left")
          .withColumn("provider_name", F.coalesce("provider_name_std", F.initcap("provider_name_clean"))))

### 4.3. Add business columns

**Why:** finance reports need the amount the TPA did not cover (`amount_not_covered`) and the share of the bill that was approved (`approval_rate`).

In [0]:
tpa = (tpa
    .withColumn("source_system", F.lit("TPA"))
    .withColumn("file_date", file_date())
    .withColumn("service_month", F.date_format("service_date", "yyyy-MM"))
    .withColumn("amount_not_covered", (F.col("billed_amount") - F.col("approved_amount")).cast("decimal(12,2)"))
    .withColumn("approval_rate",
                F.when(F.col("billed_amount") > 0,
                       F.round(F.col("approved_amount") / F.col("billed_amount"), 4))))

### 4.4. Data quality checks

**Why:** TPA business rules: the approved amount cannot be negative or higher than the billed amount, and the service cannot happen after the file was sent. Warnings (`dq_warnings`) record issues that do not make the row invalid.

In [0]:
tpa = (tpa
    .withColumn("dq_errors", issue_list(
        (F.col("tpa_ref").isNull(),                                            "MISSING_TPA_REF"),
        (F.col("_claim_id_raw").isNull(),                                      "MISSING_CLAIM_ID"),
        (F.col("_claim_id_raw").isNotNull() & F.col("claim_id").isNull(),      "INVALID_CLAIM_ID"),
        (F.col("_billed_raw").isNull(),                                        "MISSING_BILLED_AMOUNT"),
        (F.col("_billed_raw").isNotNull() & F.col("billed_amount").isNull(),   "INVALID_BILLED_AMOUNT"),
        (F.col("_approved_raw").isNull(),                                      "MISSING_APPROVED_AMOUNT"),
        (F.col("_approved_raw").isNotNull() & F.col("approved_amount").isNull(), "INVALID_APPROVED_AMOUNT"),
        ((F.col("billed_amount") < 0) | (F.col("approved_amount") < 0),         "NEGATIVE_AMOUNT"),
        (F.col("approved_amount") > F.col("billed_amount"),                     "APPROVED_MORE_THAN_BILLED"),
        (F.col("_date_raw").isNull(),                                          "MISSING_SERVICE_DATE"),
        (F.col("_date_raw").isNotNull() & F.col("service_date").isNull(),      "INVALID_SERVICE_DATE"),
        (F.col("service_date") > F.col("file_date"),                           "SERVICE_DATE_AFTER_FILE_DATE"),
    ))
    .withColumn("dq_warnings", issue_list(
        (F.col("provider_name_std").isNull() & F.col("provider_name_clean").isNotNull(), "UNKNOWN_PROVIDER"),
    )))

### 4.5. Remove duplicates

**Why:** `TPA202610120009` was sent twice in the same file: first with a negative approved amount, then corrected. Because both rows have the same file date, the row with fewer errors is kept.

In [0]:
tpa = keep_latest(tpa, "tpa_ref")

### 4.6. Final columns

**Why:** final columns and write, the same as BANK.

In [0]:
tpa_silver = finish(tpa).select(
    "source_system", "tpa_ref", "claim_id", "provider_name", "billed_amount", "approved_amount",
    "amount_not_covered", "approval_rate", "service_date", "service_month", "file_date",
    "record_versions", "dq_status", "dq_errors", "dq_warnings",
    "_source_file", "_ingested_at", "_silver_loaded_at",
)

tpa_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("insurance.silver.tpa_settlement")
display(spark.table("insurance.silver.tpa_settlement").orderBy("file_date", "tpa_ref"))

## Step 5: REPAIR → Silver

### 5.1. Clean and convert

**Why:** the same cleaning rules again. Shop names are standardised with `initcap`, because shop names have no special capitalisation (`caliber collision ` becomes `Caliber Collision`).

In [0]:
repair_bronze = spark.table("insurance.bronze.repair_invoice")

repair = repair_bronze.select(
    clean_code("invoice_no").alias("invoice_no"),
    to_int("claim_id").alias("claim_id"),
    F.initcap(clean_text("shop_name")).alias("shop_name"),
    to_amount("labor_amount").alias("labor_amount"),
    to_amount("parts_amount").alias("parts_amount_received"),
    to_amount("total_amount").alias("total_amount"),
    to_date_multi("invoice_date").alias("invoice_date"),
    clean_text("claim_id").alias("_claim_id_raw"),
    clean_text("labor_amount").alias("_labor_raw"),
    clean_text("parts_amount").alias("_parts_raw"),
    clean_text("total_amount").alias("_total_raw"),
    clean_text("invoice_date").alias("_date_raw"),
    "_source_file",
    "_ingested_at",
)

### 5.2. Fill a missing parts amount from total - labor, and add business columns

**Why:** one invoice has a blank `parts_amount`. When labor and total are present, parts can be calculated as total minus labor. The row stays valid and gets the warning `PARTS_AMOUNT_DERIVED`, so the value is marked as calculated rather than received.

In [0]:
repair = (repair
    .withColumn("parts_amount",
                F.coalesce("parts_amount_received", F.col("total_amount") - F.col("labor_amount"))
                 .cast("decimal(12,2)"))
    .withColumn("source_system", F.lit("REPAIR"))
    .withColumn("file_date", file_date())
    .withColumn("invoice_month", F.date_format("invoice_date", "yyyy-MM")))

### 5.3. Data quality checks

**Why:** repair rules: no negative amounts, and total must equal labor plus parts. A zero-amount invoice is allowed but flagged with the warning `ZERO_AMOUNT_INVOICE` for review.

In [0]:
repair = (repair
    .withColumn("dq_errors", issue_list(
        (F.col("invoice_no").isNull(),                                         "MISSING_INVOICE_NO"),
        (F.col("_claim_id_raw").isNull(),                                      "MISSING_CLAIM_ID"),
        (F.col("_claim_id_raw").isNotNull() & F.col("claim_id").isNull(),      "INVALID_CLAIM_ID"),
        (F.col("_labor_raw").isNull(),                                         "MISSING_LABOR_AMOUNT"),
        (F.col("_labor_raw").isNotNull() & F.col("labor_amount").isNull(),     "INVALID_LABOR_AMOUNT"),
        (F.col("_parts_raw").isNotNull() & F.col("parts_amount_received").isNull(), "INVALID_PARTS_AMOUNT"),
        (F.col("_total_raw").isNull(),                                         "MISSING_TOTAL_AMOUNT"),
        (F.col("_total_raw").isNotNull() & F.col("total_amount").isNull(),     "INVALID_TOTAL_AMOUNT"),
        ((F.col("labor_amount") < 0) | (F.col("parts_amount") < 0) | (F.col("total_amount") < 0), "NEGATIVE_AMOUNT"),
        (F.col("total_amount") != F.col("labor_amount") + F.col("parts_amount"), "TOTAL_MISMATCH"),
        (F.col("_date_raw").isNull(),                                          "MISSING_INVOICE_DATE"),
        (F.col("_date_raw").isNotNull() & F.col("invoice_date").isNull(),      "INVALID_INVOICE_DATE"),
    ))
    .withColumn("dq_warnings", issue_list(
        (F.col("_parts_raw").isNull() & F.col("parts_amount").isNotNull(),     "PARTS_AMOUNT_DERIVED"),
        (F.col("total_amount") == 0,                                           "ZERO_AMOUNT_INVOICE"),
    )))

### 5.4. Remove duplicates

**Why:** one row per `invoice_no`, using the same rule as the other partners.

In [0]:
repair = keep_latest(repair, "invoice_no")

### 5.5. Final columns

**Why:** final columns and write.

In [0]:
repair_silver = finish(repair).select(
    "source_system", "invoice_no", "claim_id", "shop_name", "labor_amount", "parts_amount",
    "total_amount", "invoice_date", "invoice_month", "file_date", "record_versions",
    "dq_status", "dq_errors", "dq_warnings", "_source_file", "_ingested_at", "_silver_loaded_at",
)

repair_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("insurance.silver.repair_invoice")
display(spark.table("insurance.silver.repair_invoice").orderBy("file_date", "invoice_no"))

## Step 6: Data quality summary

**Why:** one view of how many rows each partner sent and how many are valid. Expected after Day 1 and Day 2: BANK 37 rows (31 valid, 6 invalid), REPAIR 23 rows (21 valid, 2 invalid), TPA 29 rows (24 valid, 5 invalid).

In [0]:
summary = (
    spark.table("insurance.silver.bank_settlement").select("source_system", "dq_status")
    .unionByName(spark.table("insurance.silver.tpa_settlement").select("source_system", "dq_status"))
    .unionByName(spark.table("insurance.silver.repair_invoice").select("source_system", "dq_status"))
    .groupBy("source_system")
    .agg(F.count("*").alias("rows"),
         F.sum(F.when(F.col("dq_status") == "VALID", 1).otherwise(0)).alias("valid_rows"),
         F.sum(F.when(F.col("dq_status") == "INVALID", 1).otherwise(0)).alias("invalid_rows"))
    .orderBy("source_system")
)
display(summary)

## Step 7: Rows with errors or warnings

**Why:** this is the list the operations team works from. Each row shows the partner, the reference, the error and warning codes, and the source file.

In [0]:
def issue_rows(table_name, key_col):
    return (spark.table(table_name)
            .filter((F.size("dq_errors") > 0) | (F.size("dq_warnings") > 0))
            .select("source_system", F.col(key_col).alias("record_ref"), "dq_status",
                    F.array_join("dq_errors", ", ").alias("errors"),
                    F.array_join("dq_warnings", ", ").alias("warnings"),
                    "_source_file"))

all_issues = (issue_rows("insurance.silver.bank_settlement", "bank_ref")
              .unionByName(issue_rows("insurance.silver.tpa_settlement", "tpa_ref"))
              .unionByName(issue_rows("insurance.silver.repair_invoice", "invoice_no"))
              .orderBy("source_system", "record_ref"))
display(all_issues)

## Step 8: References received more than once

**Why:** shows the duplicates and updates that Silver resolved. `BNK202610110003` has 3 versions (Day 1 twice, Day 2 once) and Silver keeps the Day 2 status `RETURNED`.

In [0]:
display(spark.table("insurance.silver.bank_settlement")
        .filter(F.col("record_versions") > 1)
        .select("bank_ref", "claim_id", "amount", "status", "file_date", "record_versions"))